# Notebook 02: Merge FIA TREE, PLOT and COND Tables

This notebook merges the extracted FIA Parquet tables into one tree-level dataset for the carbon storage prediction project.

The modelling unit is one individual tree. The TREE table provides tree measurements and the carbon target, the PLOT table provides geographic and inventory context, and the COND table provides forest condition and site variables.

This notebook focuses on merging and validation only. Cleaning, missing value treatment and feature engineering will be handled in later notebooks.

## 1. Import libraries and project paths

In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.paths import PARQUET_DIR, INTERIM_DIR, OUTPUTS_DIR, create_project_dirs
from src.fia_merge import load_extracted_tables, validate_required_columns, merge_tree_plot_cond

create_project_dirs()

print("Project root:", PROJECT_ROOT)
print("Parquet folder:", PARQUET_DIR)
print("Interim folder:", INTERIM_DIR)
print("Outputs folder:", OUTPUTS_DIR)

Project root: e:\BA_DV_Project\COMM074_FIA_Project
Parquet folder: E:\BA_DV_Project\COMM074_FIA_Project\data\parquet
Interim folder: E:\BA_DV_Project\COMM074_FIA_Project\data\interim
Outputs folder: E:\BA_DV_Project\COMM074_FIA_Project\outputs


## 2. Check extracted Parquet files

In [2]:
required_parquet_files = [
    "tree_selected.parquet",
    "plot_selected.parquet",
    "cond_selected.parquet"
]

for file in required_parquet_files:
    path = PARQUET_DIR / file
    print(file, "FOUND" if path.exists() else "MISSING")

tree_selected.parquet FOUND
plot_selected.parquet FOUND
cond_selected.parquet FOUND


## 3. Load extracted TREE, PLOT and COND tables

In [3]:
tree, plot, cond = load_extracted_tables()

print("TREE shape:", tree.shape)
print("PLOT shape:", plot.shape)
print("COND shape:", cond.shape)

TREE shape: (5796627, 28)
PLOT shape: (246218, 13)
COND shape: (311972, 14)


In [4]:
tree.head()

,CN,PLT_CN,PREV_TRE_CN,INVYR,STATECD,UNITCD,COUNTYCD,PLOT,SUBP,TREE,...,CR,CCLCD,CARBON_AG,CARBON_BG,DRYBIO_AG,DRYBIO_BG,VOLCFNET,VOLCSNET,VOLBFNET,TPA_UNADJ
0,157825670010854,157531323010854,NaN,1982,1,5,111,90055,104,3,...,NaN,4.0,5.930856,1.531520,11.861712,3.063039,NaN,0.0,0.0,0.000
1,157825671010854,157531323010854,NaN,1982,1,5,111,90055,104,4,...,NaN,4.0,4.353856,1.151194,8.707712,2.302387,NaN,NaN,NaN,0.000
2,157825672010854,157531323010854,NaN,1982,1,5,111,90055,104,5,...,NaN,4.0,5.930856,1.531520,11.861712,3.063039,NaN,NaN,NaN,0.000
3,157825673010854,157531323010854,NaN,1982,1,5,111,90055,104,6,...,NaN,4.0,3.889810,1.038430,7.779619,2.076860,NaN,NaN,NaN,0.000
4,157825674010854,157531324010854,NaN,1982,1,5,111,90056,101,1,...,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,91.685


In [5]:
plot.head()

,CN,STATECD,UNITCD,COUNTYCD,PLOT,PLOT_STATUS_CD,INVYR,MEASYEAR,MEASMON,MEASDAY,LAT,LON,ELEV
0,44540625020004,41,4,45,92194,2.0,2013,2013.0,9.0,1.0,42.223685,-117.701166,5100.0
1,44541690020004,41,4,45,93235,2.0,2013,2013.0,9.0,1.0,43.556785,-117.167603,3300.0
2,44540333020004,41,4,45,93739,2.0,2013,2013.0,9.0,1.0,44.005172,-117.029425,2100.0
3,44543226020004,41,4,45,95160,2.0,2013,2013.0,9.0,1.0,43.360484,-117.168441,4100.0
4,44541685020004,41,4,45,95633,2.0,2013,2013.0,9.0,1.0,42.672177,-117.581170,4100.0


In [6]:
cond.head()

,CN,PLT_CN,CONDID,COND_STATUS_CD,FORTYPCD,FLDTYPCD,OWNGRPCD,RESERVCD,SITECLCD,STDAGE,STDSZCD,BALIVE,ALSTK,GSSTK
0,205297170010854,43539075010478,1,1,161.0,161.0,40.0,0.0,5.0,31.0,2.0,113.4604,62.5064,45.9191
1,205297171010854,43539075010478,2,1,520.0,520.0,40.0,0.0,4.0,30.0,2.0,38.5134,28.0173,26.6401
2,205297185010854,43540262010478,1,1,520.0,406.0,40.0,0.0,4.0,25.0,2.0,48.9767,36.1633,18.1259
3,205297182010854,43540098010478,1,1,161.0,161.0,40.0,0.0,5.0,31.0,1.0,63.2295,39.5035,39.0869
4,205297183010854,43540098010478,2,1,404.0,406.0,40.0,0.0,5.0,15.0,2.0,66.5839,36.9382,26.3756


## 4. Validate required columns before merging

In [7]:
required_tree_cols = ["CN", "PLT_CN", "CONDID", "TREE", "STATUSCD", "SPCD", "DIA", "HT", "ACTUALHT", "CR", "CARBON_AG"]
required_plot_cols = ["CN", "STATECD", "COUNTYCD", "INVYR"]
required_cond_cols = ["PLT_CN", "CONDID", "FORTYPCD", "OWNGRPCD", "SITECLCD", "STDAGE", "STDSZCD"]

missing_tree = validate_required_columns(tree, required_tree_cols, "TREE")
missing_plot = validate_required_columns(plot, required_plot_cols, "PLOT")
missing_cond = validate_required_columns(cond, required_cond_cols, "COND")

print("Missing TREE columns:", missing_tree)
print("Missing PLOT columns:", missing_plot)
print("Missing COND columns:", missing_cond)

TREE has all required columns.
PLOT has all required columns.
COND has all required columns.
Missing TREE columns: []
Missing PLOT columns: []
Missing COND columns: []


## 5. Check join keys before merging

In [8]:
print("TREE rows:", len(tree))
print("Unique TREE CN:", tree["CN"].nunique())

print("\nPLOT rows:", len(plot))
print("Unique PLOT CN:", plot["CN"].nunique())

print("\nCOND rows:", len(cond))
print("Unique COND PLT_CN + CONDID pairs:", cond[["PLT_CN", "CONDID"]].drop_duplicates().shape[0])

TREE rows: 5796627
Unique TREE CN: 5796627

PLOT rows: 246218
Unique PLOT CN: 246218

COND rows: 311972
Unique COND PLT_CN + CONDID pairs: 311972


In [9]:
duplicate_plot_keys = plot["CN"].duplicated().sum()
duplicate_cond_keys = cond[["PLT_CN", "CONDID"]].duplicated().sum()

print("Duplicate PLOT CN keys:", duplicate_plot_keys)
print("Duplicate COND PLT_CN + CONDID keys:", duplicate_cond_keys)

Duplicate PLOT CN keys: 0
Duplicate COND PLT_CN + CONDID keys: 0


## 6. Merge TREE, PLOT and COND tables

In [10]:
merged_df, merge_summary = merge_tree_plot_cond()

merge_summary

TREE has all required columns.
PLOT has all required columns.
COND has all required columns.
TREE rows before merge: 5,796,627
PLOT rows before merge: 246,218
COND rows before merge: 311,972
Rows after TREE-PLOT merge: 5,796,627
Rows after COND merge: 5,796,627
Merged dataset saved to E:\BA_DV_Project\COMM074_FIA_Project\data\interim\merged_tree_plot_cond.parquet
Merge summary saved to E:\BA_DV_Project\COMM074_FIA_Project\outputs\merge_summary.csv
Merged column summary saved to E:\BA_DV_Project\COMM074_FIA_Project\outputs\merged_column_summary.csv


,metric,value
0,tree_rows_before_merge,5796627
1,plot_rows_before_merge,246218
2,cond_rows_before_merge,311972
3,rows_after_tree_plot_merge,5796627
4,rows_after_cond_merge,5796627
5,final_row_count_matches_tree,True
6,missing_plot_match_rate,0.0
7,missing_cond_match_rate,0.015009
8,plot_statecd_missing_rate,0.0
9,cond_fortypcd_missing_rate,0.015009


## 7. Validate merged dataset

In [11]:
print("Original TREE rows:", len(tree))
print("Merged rows:", len(merged_df))
print("Row count difference:", len(merged_df) - len(tree))

Original TREE rows: 5796627
Merged rows: 5796627
Row count difference: 0


In [12]:
key_plot_fields = [c for c in ["STATECD", "COUNTYCD", "INVYR"] if c in merged_df.columns]
key_cond_fields = [c for c in ["FORTYPCD", "OWNGRPCD", "SITECLCD", "STDAGE", "STDSZCD"] if c in merged_df.columns]

print("Missing rates for key PLOT fields:")
display(merged_df[key_plot_fields].isna().mean().mul(100).round(2))

print("Missing rates for key COND fields:")
display(merged_df[key_cond_fields].isna().mean().mul(100).round(2))

Missing rates for key PLOT fields:


STATECD     0.0
COUNTYCD    0.0
INVYR       0.0
dtype: float64

Missing rates for key COND fields:


FORTYPCD    1.50
OWNGRPCD    0.82
SITECLCD    1.50
STDAGE      2.20
STDSZCD     1.68
dtype: float64

In [13]:
important_cols = [
    "CN", "PLT_CN", "CONDID", "TREE",
    "DIA", "HT", "ACTUALHT", "CR",
    "SPCD", "CARBON_AG",
    "STATECD", "COUNTYCD", "INVYR",
    "FORTYPCD", "OWNGRPCD", "SITECLCD", "STDAGE", "STDSZCD"
]

existing_important_cols = [c for c in important_cols if c in merged_df.columns]
missing_important_cols = [c for c in important_cols if c not in merged_df.columns]

print("Existing important columns:", existing_important_cols)
print("Missing important columns:", missing_important_cols)

Existing important columns: ['CN', 'PLT_CN', 'CONDID', 'TREE', 'DIA', 'HT', 'ACTUALHT', 'CR', 'SPCD', 'CARBON_AG', 'STATECD', 'COUNTYCD', 'INVYR', 'FORTYPCD', 'OWNGRPCD', 'SITECLCD', 'STDAGE', 'STDSZCD']
Missing important columns: []


In [14]:
merged_df[existing_important_cols].head()

,CN,PLT_CN,CONDID,TREE,DIA,HT,ACTUALHT,CR,SPCD,CARBON_AG,STATECD,COUNTYCD,INVYR,FORTYPCD,OWNGRPCD,SITECLCD,STDAGE,STDSZCD
0,157825670010854,157531323010854,1,3,2.6,NaN,NaN,NaN,110,5.930856,1,111,1982,162.0,40.0,5.0,10.0,3.0
1,157825671010854,157531323010854,1,4,2.3,NaN,NaN,NaN,131,4.353856,1,111,1982,162.0,40.0,5.0,10.0,3.0
2,157825672010854,157531323010854,1,5,2.6,NaN,NaN,NaN,131,5.930856,1,111,1982,162.0,40.0,5.0,10.0,3.0
3,157825673010854,157531323010854,1,6,2.2,NaN,NaN,NaN,131,3.889810,1,111,1982,162.0,40.0,5.0,10.0,3.0
4,157825674010854,157531324010854,1,1,2.9,NaN,NaN,NaN,400,0.000000,1,111,1982,406.0,40.0,4.0,5.0,3.0


In [15]:
merged_df[existing_important_cols].info()

<class 'pandas.DataFrame'>
RangeIndex: 5796627 entries, 0 to 5796626
Data columns (total 18 columns):
 #   Column     Dtype  
---  ------     -----  
 0   CN         int64  
 1   PLT_CN     int64  
 2   CONDID     int64  
 3   TREE       int64  
 4   DIA        float64
 5   HT         str    
 6   ACTUALHT   str    
 7   CR         float64
 8   SPCD       int64  
 9   CARBON_AG  float64
 10  STATECD    int64  
 11  COUNTYCD   int64  
 12  INVYR      int64  
 13  FORTYPCD   float64
 14  OWNGRPCD   float64
 15  SITECLCD   float64
 16  STDAGE     float64
 17  STDSZCD    float64
dtypes: float64(8), int64(8), str(2)
memory usage: 831.2 MB


In [16]:
merged_df["STATECD"].value_counts().sort_index()

STATECD
1     1115757
6      446320
13    1880767
23    1030866
41     791427
53     531490
Name: count, dtype: int64

## 8. Merge summary

The extracted TREE, PLOT and COND tables were merged to create a tree-level dataset. The TREE table was treated as the base table because the project predicts individual-tree above-ground carbon storage. PLOT information was joined using the plot identifier, while COND information was joined using both plot identifier and condition identifier.

The row-count validation confirms whether the merge preserved the original number of tree records. This is important because an incorrect join could duplicate or remove tree records, which would bias later EDA and modelling.

## 9. Summary

This notebook merged the selected FIA TREE, PLOT and COND extracts into one tree-level dataset. The merge process preserved the tree-level unit of analysis and validated row counts before and after each join.

The merged dataset includes tree measurements, the above-ground carbon target, geographic context and forest condition variables. It has been saved as:

- `data/interim/merged_tree_plot_cond.parquet`

The next notebook will focus on data cleaning, missing value analysis and biological validity checks.